# Stage 1 — Segmentation (Mesmer)

**Input**: Raw TIFF images (defined in `config.yaml` per machine)  
**Output**: Labeled TIFF + overlay PNG per ROI → `<experiment>/stage1_segmentation/<machine>/<roi>/`

This stage runs DeepCell Mesmer segmentation on all ROIs across all machines.

In [2]:
import os
import numpy as np
from pathlib import Path
from skimage.io import imread
from matplotlib import pyplot as plt
from tqdm import tqdm

from pipeline.config import load_config

# Load configuration
cfg = load_config("config.yaml")
print(f"Experiment: {cfg.experiment_name}")
print(f"Machines: {list(cfg.machines.keys())}")

ImportError: cannot import name 'broadcast_to' from 'numpy.lib.stride_tricks' (/home/hp/projects/multiplex_imaging_analysis/.venv/lib/python3.10/site-packages/numpy/lib/stride_tricks.py)

In [ ]:
from deepcell.applications import Mesmer
from deepcell.utils.plot_utils import create_rgb_image, make_outline_overlay

# Initialise Mesmer (downloads model on first run)
app = Mesmer()

In [ ]:
def prepare_image(file_name, dapi=False):
    """Load and stack nuclear + membrane channels into 4D array."""
    if not dapi:
        img_nuclei = imread(f"{file_name}_nuclei.tif")
        img_membrane = imread(f"{file_name}_membrane.tif")
    else:
        img_nuclei = imread(file_name)
        img_membrane = imread(file_name)
    img = np.stack((img_nuclei, img_membrane), axis=-1)
    img = np.expand_dims(img, 0)
    return img


def segment_cells(file_name, image_mpp=0.176, params=None, dapi=False):
    """Segment cells using Mesmer. Returns (overlay, labeled_image)."""
    if params is None:
        params = cfg.segmentation.postprocess_kwargs
    img = prepare_image(file_name, dapi)
    rgb_img = create_rgb_image(img, channel_colors=["green", "blue"])
    labeled_image = app.predict(
        img, image_mpp=image_mpp, postprocess_kwargs_whole_cell=params
    )
    overlay_data = make_outline_overlay(rgb_data=rgb_img, predictions=labeled_image)
    return overlay_data, labeled_image

## Run Segmentation

Process all ROIs for all machines. Outputs saved to `stage1_segmentation/<machine>/<roi>/`.

In [ ]:
from PIL import Image

seg_params = cfg.segmentation.postprocess_kwargs
image_mpp = cfg.segmentation.image_mpp
dapi_only = cfg.segmentation.dapi_only

for machine_name, machine_cfg in cfg.machines.items():
    tiff_dir = machine_cfg.raw_tiff_dir
    if not tiff_dir or not os.path.isdir(tiff_dir):
        print(f"⚠ Skipping {machine_name}: raw_tiff_dir not found ({tiff_dir})")
        continue

    output_dir = cfg.ensure_stage_dir(1, machine_name)
    print(f"\n{'='*60}")
    print(f"Machine: {machine_name}")
    print(f"Input:   {tiff_dir}")
    print(f"Output:  {output_dir}")
    print(f"{'='*60}")

    for file_name in tqdm(sorted(os.listdir(tiff_dir))):
        file_path = os.path.join(tiff_dir, file_name)
        if not file_name.lower().endswith((".tif", ".tiff")):
            continue

        base, _ = os.path.splitext(file_name)
        roi_dir = output_dir / base
        roi_dir.mkdir(parents=True, exist_ok=True)

        try:
            overlay, labeled = segment_cells(
                file_path, image_mpp=image_mpp, params=seg_params, dapi=dapi_only
            )
            # Save labeled TIFF
            img_out = Image.fromarray(labeled[0][:, :, 0].astype(np.int16))
            img_out.save(roi_dir / "labeled.tif")
            # Save overlay PNG
            plt.imsave(str(roi_dir / "overlay.png"), overlay[0])
            # Save labeled preview
            plt.imsave(str(roi_dir / "labeled_preview.png"), labeled[0][:, :, 0])
            print(f"  ✓ {base}")
        except Exception as e:
            print(f"  ✗ {base}: {e}")

print("\n✓ Stage 1 complete.")